# Agent 1 — SOP Compliance (Automotive Service-Bay Scenario)

> Per-arrival audit: was the technician greeting within 60 seconds of
> the vehicle parking? Was the air filter inspected? Same code as the
> QSR drive-thru scenario — different search query, different VLM prompt.

Compare to `01_qsr_drivethru_sop.ipynb` to see how generic the SOP
Compliance pattern is. The whole "agent" is `(search_query, vlm_prompt,
time_window)`.


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


In [ ]:
def search(query, *, video_nos=None, unique_id="default", top_k=10,
           filtering_level="medium", search_type="BY_CLIP",
           datetime_taken=None, tag=None, camera_tag=None,
           latitude=None, longitude=None, max_retries=3):
    """Visual Search — POST /search.

    Returns the list of {videoNo, startTime, endTime, score, ...} matches.

    Filter parameters (all optional, combinable):
      • video_nos       restrict to specific videos (up to 100 ids)
      • datetime_taken  filter to videos captured at-or-after this timestamp
      • tag             filter to videos carrying this user-defined tag
      • camera_tag      filter to videos shot on this camera_model
      • latitude/longitude  GPS proximity filter (must be paired)

    Notes on the envelope: Visual Search responses are wrapped in
    {code, msg, data, success, failed}. code="0000" means OK; the actual
    hit list is in `data`. code="0001" ("network abnormal") is transient
    and we retry it.
    """
    body = {
        "search_param": query,
        "search_type": search_type,                 # "BY_CLIP" or "BY_AUDIO"
        "unique_id": unique_id,                      # namespace within your account
        "top_k": top_k,
        "filtering_level": filtering_level,          # "low"|"medium"|"high"
    }
    if video_nos:        body["video_nos"]     = list(video_nos)
    if datetime_taken:   body["datetime_taken"] = datetime_taken
    if tag:              body["tag"]            = tag
    if camera_tag:       body["camera_tag"]     = camera_tag
    if latitude is not None and longitude is not None:
        body["latitude"]  = latitude
        body["longitude"] = longitude

    for attempt in range(max_retries):
        r = requests.post(f"{VS_HOST}/search", headers=HEADERS, json=body, timeout=60)
        r.raise_for_status()
        envelope = r.json()
        code = envelope.get("code")
        if code == "0000":
            return envelope.get("data") or []
        if code == "0001" and attempt < max_retries - 1:
            # Transient. Backoff and retry.
            time.sleep(0.4 * (2 ** attempt))
            continue
        raise RuntimeError(f"/search failed: code={code} msg={envelope.get('msg')!r}")
    return []


In [ ]:
def vlm_complete(prompt, *, video_url=None, image_url=None, system=None,
                 model=VLM_MODEL, response_json=True, temperature=0.2,
                 max_tokens=1024):
    """Visual Intelligence — POST /vu/chat/completions.

    Calls a Video Language Model (Gemini by default) with text + an
    optional media reference. Returns the assistant's text reply, with
    markdown ```json fences stripped if `response_json=True`.

    Notes on the wire format:
      • `content` MUST be an array even for text-only prompts. A bare
        string is rejected with "Model input cannot be empty".
      • Gemini's response lives in choices[0]["text"]. Other providers
        (Qwen, Nova) use choices[0]["message"]["content"]. We accept both.
      • The endpoint can return HTTP 200 with status="errored" — surface
        that as a typed exception rather than letting JSON parsing fail.
    """
    content = [{"type": "text", "text": prompt}]
    if video_url:
        content.append({"type": "input_file", "file_uri": video_url, "mime_type": "video/mp4"})
    if image_url:
        content.append({"type": "input_file", "file_uri": image_url, "mime_type": "image/jpeg"})

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": content})

    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if response_json:
        # Tells Gemini to bias toward JSON output. Other providers ignore this.
        body["extra_body"] = {"metadata": {"response_mime_type": "application/json"}}

    r = requests.post(f"{VLM_HOST}/vu/chat/completions", headers=HEADERS,
                      json=body, timeout=180)
    r.raise_for_status()
    envelope = r.json()
    if envelope.get("status") == "errored" or envelope.get("error"):
        err = envelope.get("error") or {}
        raise RuntimeError(f"VLM error: {err.get('code')} {err.get('message')}")
    choices = envelope.get("choices") or []
    if not choices:
        raise RuntimeError(f"VLM returned no choices: {envelope}")
    # Two shapes observed in the wild.
    text = choices[0].get("text") or (choices[0].get("message") or {}).get("content", "")
    if response_json:
        text = _strip_json_fence(text)
    return text


def _strip_json_fence(text):
    """Gemini often wraps JSON output in ```json ... ``` fences even when
    response_mime_type=application/json. Trim them so json.loads() works."""
    if not text:
        return text
    s = text.strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s[3:]
        if s.endswith("```"):
            s = s[:-3].rstrip()
    return s


In [ ]:
def resolve_media_url(video_no):
    """Bridge a videoNo to a public URL the VLM can fetch.

    `/download` streams the raw bytes back to you — it does NOT return a
    hosted URL. To feed a video to /vu/chat/completions you must host it
    yourself. This helper expects either:

      • MEMORIES_MEDIA_URL_TEMPLATE env var (e.g. https://your-cdn/{video_no}.mp4)
      • a MEDIA_URL_MAP dict you populate inline

    If neither is configured, raises so the notebook stops cleanly.
    """
    if video_no in MEDIA_URL_MAP:
        return MEDIA_URL_MAP[video_no]
    tpl = os.environ.get("MEMORIES_MEDIA_URL_TEMPLATE")
    if tpl:
        return tpl.format(video_no=video_no)
    raise RuntimeError(
        f"No media-URL bridge configured for {video_no}. "
        "Set MEMORIES_MEDIA_URL_TEMPLATE or add an entry to MEDIA_URL_MAP."
    )

# Per-notebook overrides: populate this for testing without a CDN.
# A public Memories.ai test asset is included as an example.
MEDIA_URL_MAP = {
    # "VI676024023022092288": "https://storage.googleapis.com/memories-test-data/test_1min.mp4",
}


## Step 1 — detect vehicle arrivals

Each arrival is the start of a 60-second audit window.


In [ ]:
seed_hits = search("a person", top_k=1, filtering_level=None)
VIDEO_NO  = seed_hits[0]["videoNo"]

arrivals = search(
    "car drives into service bay and parks",
    video_nos=[VIDEO_NO], top_k=10, filtering_level="medium",
)

# If no vehicle-arrival hits (likely when the seed isn't service-bay
# footage), fall back to a broad query so the notebook still
# demonstrates the per-arrival VLM audit. In production you'd point
# the agent at your own service-bay footage and skip this fallback.
if not arrivals:
    print("(no vehicle-arrival hits — falling back to broad query for demo)")
    arrivals = search("a person", video_nos=[VIDEO_NO], top_k=3,
                      filtering_level=None)

print(f"Detected {len(arrivals)} arrival(s)")
for a in arrivals:
    print(f"  arrival @ {a['startTime']}s  score={a['score']:.3f}")


## Step 2 — VLM audit per arrival

For each arrival, ask the VLM about the 60 seconds *after* the vehicle
appeared: was there a greeting? How long until greeting? Air filter
inspected?


In [ ]:
MEDIA_URL_MAP[VIDEO_NO] = "https://storage.googleapis.com/memories-test-data/test_1min.mp4"
media_url = resolve_media_url(VIDEO_NO)

WINDOW_SEC       = 60.0   # how long after arrival to inspect
GREET_TARGET_SEC = 60.0   # SOP target: greeting must happen within this many seconds

SYSTEM = ("You audit automotive service-bay SOPs from camera footage. "
          "Reply strict JSON only, matching the user-provided schema.")
SCHEMA = ('{"technician_greeted": bool, "time_to_greet_sec": int|null, '
          '"air_filter_checked": bool, "tire_pressure_checked": bool, "notes": str}')

audits = []
for i, a in enumerate(arrivals):
    start = float(a["startTime"])
    end   = start + WINDOW_SEC
    prompt = (
        f"Between {start:.0f}s and {end:.0f}s a vehicle just arrived in the "
        "service bay. Did a technician greet the vehicle? How many seconds "
        "elapsed before greeting (null if no greeting)? Was an air filter "
        "inspected? Was tire pressure checked? "
        f"Reply JSON only, matching: {SCHEMA}"
    )
    raw = vlm_complete(prompt, video_url=media_url, system=SYSTEM)
    try:
        verdict = json.loads(raw)
    except json.JSONDecodeError:
        verdict = {"raw": raw, "parse_error": True}
    audits.append({"arrival_t": start, "verdict": verdict})
    print(f"[{i+1}/{len(arrivals)}] arrival @ {start:.0f}s → {verdict}")


## Step 3 — aggregate compliance

Count arrivals that were greeted within the target.


In [ ]:
compliant = sum(
    1 for a in audits
    if isinstance(a["verdict"], dict)
    and a["verdict"].get("technician_greeted")
    and (a["verdict"].get("time_to_greet_sec") or 1e9) <= GREET_TARGET_SEC
)
print(f"\n{compliant}/{len(audits)} arrivals greeted within {GREET_TARGET_SEC:.0f}s.")
